# Dependency imports

In [1]:
import torch
from torchvision.models import convnext_base

from Training.training import Config, TrainingResources, initClassifierTrainers, trainingLoop
import Training.metrics as m
import Training.plotting as plt

# *modelname* specific configuration

eg.: ResNet-152

In [2]:
c = Config("ConvNext-Base")

#Configure execution to needs
c.purge    = False
c.training = True
c.loadBest = False

#Set hyperparams to model
c.learningRate = 1e-4
c.weightDecay  = 1e-3

#c.numClasses = 8
#c.batchSize = 4
#c.workers = 8


# Initialize resources

In [3]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

model = convnext_base(weights=None)
model.classifier[2] = torch.nn.Linear(model.classifier[2].in_features, out_features=c.numClasses)

TR = TrainingResources(c, model, initClassifierTrainers, device)
TR.load(c)

print(TR.criterion)
print(TR.optimizer)

#Note: After this point only some config changes will apply

/home/csuti/Development/Medical_Data_Processing/venv/lib/python3.12/site-packages/torch/cuda/__init__.py:184: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


Using cpu device
CrossEntropyLoss()
AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0001
    maximize: False
    weight_decay: 0.001
)


# Execute training

In [4]:
# Override training related config if neccesary
c.training = True
c.gpuSleep = 10
c.trainingEpochs = 20

trainingLoop(c, TR, stopAt=3)

display(TR.history)

Epoch 1/3 	| 

KeyboardInterrupt: 

# Plots of learning process

# Metrics of trained model